# Colab/Local Evaluation + Report Runbook

Orchestrates the official EGNN-vs-HGNN-vs-constant-velocity comparison report end to end. Reuses the existing CLIs (`evaluation.evaluate`, `evaluation.evaluate_baseline`, `evaluation.report`, `evaluation.animate_best`) and writes every artifact under one report-scoped directory so the run is fully self-contained.

## Before running

1. Set `RUN_ENV` below to `"colab"` or `"local"`.
2. In Colab: select a GPU runtime, paste a GitHub token, and put `train.h5` / `test.h5` under `MyDrive/masters-thesis/data/output/`.
3. Set `EGNN_CHECKPOINT` and `HGNN_CHECKPOINT` to the official 1k run checkpoints.

## Output layout

```text
{REPORT_DIR}/
    evaluations/
        egnn/metrics.json
        hgnn/metrics.json
        baseline_constant_velocity/metrics.json
    figures/    # rollout MSE, energy drift, per-horizon snapshots
    tables/     # per_bin_summary.csv, key_timestep_summary.csv
    report.md
    animations/ # optional, populated when RUN_ANIMATIONS is True
```


In [ ]:
# @title 1. Setup: environment switch, Drive mount, repo clone
import subprocess
from pathlib import Path

RUN_ENV = "local"  # @param ["local", "colab"]
GITHUB_TOKEN = ""  # @param {type:"string"}
REPO = "AlexNeagu123/msc-thesis-gnn-nbody"
GIT_REF = "main"  # @param {type:"string"}


def _shell(*cmd: str) -> None:
    """Stream a subprocess to the cell output and raise on non-zero exit."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if proc.stdout is not None:
        for line in proc.stdout:
            print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


if RUN_ENV == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/masters-thesis")

    if not GITHUB_TOKEN:
        raise ValueError("Paste a GitHub token before cloning the repo.")
    repo_dir = Path("/content/repo")
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"
    if (repo_dir / ".git").exists():
        _shell("git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", GIT_REF)
        _shell("git", "-C", str(repo_dir), "checkout", "FETCH_HEAD")
    elif repo_dir.exists():
        raise FileExistsError(f"{repo_dir} exists but is not a git checkout")
    else:
        _shell("git", "clone", "--depth", "1", "--branch", GIT_REF, clone_url, str(repo_dir))
    project_dir = (
        repo_dir / "impl" if (repo_dir / "impl" / "requirements.txt").exists() else repo_dir
    )
    get_ipython().run_line_magic("cd", str(project_dir))
    _shell("pip", "install", "-q", "-r", "requirements.txt")
elif RUN_ENV == "local":
    DRIVE = None
    project_dir = Path.cwd()
    # Make sure the cell's CWD is the impl/ root no matter where the notebook was opened from.
    if not (project_dir / "evaluation" / "report.py").exists():
        candidate = project_dir / "impl"
        if (candidate / "evaluation" / "report.py").exists():
            get_ipython().run_line_magic("cd", str(candidate))
            project_dir = candidate
        else:
            raise FileNotFoundError(
                f"Could not locate impl/ under {project_dir}; open the notebook from the repo root."
            )
else:
    raise ValueError(f"Unknown RUN_ENV={RUN_ENV!r}; choose 'local' or 'colab'.")

print(f"RUN_ENV={RUN_ENV}")
print(f"project_dir={project_dir}")


In [ ]:
# @title 2. User parameters
REPORT_NAME = "official_1k"  # @param {type:"string"}
EGNN_CHECKPOINT = "runs/egnn/20260510_211102/best.pt"  # @param {type:"string"}
HGNN_CHECKPOINT = "runs/hgnn/20260510_211151/best.pt"  # @param {type:"string"}
EGNN_CONFIG = "configs/egnn.yaml"  # @param {type:"string"}
HGNN_CONFIG = "configs/hgnn.yaml"  # @param {type:"string"}
TRAIN_PATH = "data/output/train.h5"  # @param {type:"string"}
TEST_PATH = "data/output/test.h5"  # @param {type:"string"}
DEVICE = "cuda" if RUN_ENV == "colab" else "auto"  # noqa: F821 - RUN_ENV defined in cell 1
FORCE_EVALUATION = False  # @param {type:"boolean"}
FORCE_DATA_COPY = False  # @param {type:"boolean"}
RUN_ANIMATIONS = False  # @param {type:"boolean"}
ANIMATION_FPS = 20  # @param {type:"integer"}

if RUN_ENV == "colab":
    REPORT_DIR = DRIVE / "runs" / "reports" / REPORT_NAME
else:
    REPORT_DIR = Path("runs/reports") / REPORT_NAME

EVAL_DIRS = {
    "egnn": REPORT_DIR / "evaluations" / "egnn",
    "hgnn": REPORT_DIR / "evaluations" / "hgnn",
    "baseline": REPORT_DIR / "evaluations" / "baseline_constant_velocity",
}
METRICS_PATHS = {name: eval_dir / "metrics.json" for name, eval_dir in EVAL_DIRS.items()}

REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"REPORT_DIR={REPORT_DIR}")
print(f"DEVICE={DEVICE}")
print(f"FORCE_EVALUATION={FORCE_EVALUATION}")
print(f"RUN_ANIMATIONS={RUN_ANIMATIONS}")


In [ ]:
# @title 3. Preflight: check checkpoints + configs + data exist
import shutil

import h5py


def _require_file(path: str, label: str, hint: str = "") -> None:
    """Fail fast at the orchestrator boundary with a path-specific hint."""
    if Path(path).exists():
        print(f"[ok]      {label}: {path}")
        return
    msg = f"missing {label}: {path}"
    if hint:
        msg = f"{msg}\n  {hint}"
    raise FileNotFoundError(msg)


def _ensure_dataset(local_path: str, drive_subpath: str) -> None:
    """Local mode asserts existence; Colab copies from Drive (and re-copies on FORCE_DATA_COPY).

    `FORCE_DATA_COPY=True` deletes the local copy before pulling from Drive
    so a regenerated official dataset can refresh stale local h5 files.
    Outside Colab the toggle is a no-op because there is no canonical source
    to refresh from.
    """
    path = Path(local_path)
    if path.exists() and not (FORCE_DATA_COPY and RUN_ENV == "colab"):
        print(f"[ok]      data: {path}")
        return
    if RUN_ENV != "colab":
        raise FileNotFoundError(f"Missing data file: {local_path}")
    src = DRIVE / drive_subpath
    if not src.exists():
        raise FileNotFoundError(f"Missing both local {path} and Drive source {src}")
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    shutil.copy2(src, path)
    print(f"[copied]  data: {src} -> {path}")


_colab_ckpt_hint = (
    "In Colab the cloned repo has no runs/ tree; point this at a Drive path like "
    "/content/drive/MyDrive/masters-thesis/runs/<model>/<run_id>/best.pt."
    if RUN_ENV == "colab"
    else "Update the checkpoint path in cell 2 to a real best.pt file."
)

_require_file(EGNN_CHECKPOINT, "EGNN_CHECKPOINT", hint=_colab_ckpt_hint)
_require_file(HGNN_CHECKPOINT, "HGNN_CHECKPOINT", hint=_colab_ckpt_hint)
_require_file(EGNN_CONFIG, "EGNN_CONFIG")
_require_file(HGNN_CONFIG, "HGNN_CONFIG")

_ensure_dataset(TRAIN_PATH, "data/output/train.h5")
_ensure_dataset(TEST_PATH, "data/output/test.h5")

for label, path in (("train", TRAIN_PATH), ("test", TEST_PATH)):
    with h5py.File(path, "r") as f:
        print(f"{label}: trajectories shape = {f['trajectories'].shape}")


In [ ]:
# @title 4. Evaluate EGNN and HGNN checkpoints
def _evaluate(name: str, config: str, checkpoint: str, output_dir: Path) -> None:
    metrics_path = output_dir / "metrics.json"
    if metrics_path.exists() and not FORCE_EVALUATION:
        print(f"skipping {name} eval: {metrics_path} exists (FORCE_EVALUATION=False)")
        return
    output_dir.mkdir(parents=True, exist_ok=True)
    _shell(
        "python",
        "-u",
        "-m",
        "evaluation.evaluate",
        "--config",
        config,
        "--checkpoint",
        checkpoint,
        "--test-path",
        TEST_PATH,
        "--output-dir",
        str(output_dir),
        "--device",
        DEVICE,
    )
    if not metrics_path.exists():
        raise RuntimeError(f"{name} eval finished but {metrics_path} is missing")


_evaluate("egnn", EGNN_CONFIG, EGNN_CHECKPOINT, EVAL_DIRS["egnn"])
_evaluate("hgnn", HGNN_CONFIG, HGNN_CHECKPOINT, EVAL_DIRS["hgnn"])


In [ ]:
# @title 5. Evaluate the constant-velocity baseline
def _evaluate_baseline() -> None:
    metrics_path = METRICS_PATHS["baseline"]
    if metrics_path.exists() and not FORCE_EVALUATION:
        print(f"skipping baseline eval: {metrics_path} exists (FORCE_EVALUATION=False)")
        return
    EVAL_DIRS["baseline"].mkdir(parents=True, exist_ok=True)
    _shell(
        "python",
        "-u",
        "-m",
        "evaluation.evaluate_baseline",
        "--baseline",
        "constant_velocity",
        "--train-path",
        TRAIN_PATH,
        "--test-path",
        TEST_PATH,
        "--output-dir",
        str(EVAL_DIRS["baseline"]),
        "--device",
        DEVICE,
    )
    if not metrics_path.exists():
        raise RuntimeError(f"baseline eval finished but {metrics_path} is missing")


_evaluate_baseline()


In [ ]:
# @title 6. Generate the comparison report
_shell(
    "python",
    "-u",
    "-m",
    "evaluation.report",
    "--egnn",
    str(METRICS_PATHS["egnn"]),
    "--hgnn",
    str(METRICS_PATHS["hgnn"]),
    "--baseline",
    str(METRICS_PATHS["baseline"]),
    "--output",
    str(REPORT_DIR),
)

report_md = REPORT_DIR / "report.md"
figures_dir = REPORT_DIR / "figures"
tables_dir = REPORT_DIR / "tables"
print()
print(f"report.md : {report_md}")
print(f"figures   : {figures_dir} ({len(list(figures_dir.glob('*.png')))} PNG)")
print(f"tables    : {tables_dir} ({len(list(tables_dir.glob('*.csv')))} CSV)")


In [ ]:
# @title 7. Optional: render best-trajectory animations
# Slow: re-runs both model rollouts to drive the FuncAnimation loop. Toggle RUN_ANIMATIONS
# in cell 2 only when you actually want the per-bin videos refreshed.
if not RUN_ANIMATIONS:
    print("RUN_ANIMATIONS=False; skipping animation rendering.")
else:
    ANIMATIONS_DIR = REPORT_DIR / "animations"
    ANIMATIONS_DIR.mkdir(parents=True, exist_ok=True)
    _shell(
        "python",
        "-u",
        "-m",
        "evaluation.animate_best",
        "--egnn-checkpoint",
        EGNN_CHECKPOINT,
        "--hgnn-checkpoint",
        HGNN_CHECKPOINT,
        "--egnn-config",
        EGNN_CONFIG,
        "--hgnn-config",
        HGNN_CONFIG,
        "--test-path",
        TEST_PATH,
        "--output-dir",
        str(ANIMATIONS_DIR),
        "--device",
        DEVICE,
        "--fps",
        str(ANIMATION_FPS),
    )
    gifs = sorted(ANIMATIONS_DIR.glob("*.gif"))
    mp4s = sorted(ANIMATIONS_DIR.glob("*.mp4"))
    print()
    print(f"animations: {ANIMATIONS_DIR}")
    print(f"  GIFs: {len(gifs)}")
    print(f"  MP4s: {len(mp4s)} (zero is expected when ffmpeg is unavailable)")
